# 03 · Index
Builds the two search indexes over `data/chunks/chunks.jsonl` (from 02):
- **Dense:** chunk embeddings from an Ollama embedding model, stored in Chroma (`data/chroma`, cosine distance), with the chunk metadata.
- **Sparse:** BM25 (`rank_bm25`) over the same chunks, saved to `data/bm25/bm25.pkl`.

Embeddings are cached in `data/embed_cache/`. The cache key is source file hash + embedding model + chunk settings hash, so changing any of them never reuses stale vectors.

The embedder table also lists bge-m3 and nomic-embed-text for the comparison in 04; this notebook builds the default one.

## 1 · Imports and settings

In [1]:
import hashlib, json, pickle, random, re, time
from collections import Counter
from pathlib import Path

import chromadb
import numpy as np
import requests
from rank_bm25 import BM25Okapi

DATA_DIR = Path("../data")
CHUNKS_FILE = DATA_DIR / "chunks" / "chunks.jsonl"
CHUNK_SETTINGS_FILE = DATA_DIR / "chunks" / "settings.json"
CACHE_DIR = DATA_DIR / "embed_cache"
CHROMA_DIR = DATA_DIR / "chroma"
BM25_FILE = DATA_DIR / "bm25" / "bm25.pkl"
for d in (CACHE_DIR, CHROMA_DIR, BM25_FILE.parent):
    d.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://localhost:11434"
EMBED_BATCH = 16
EVIDENCE_ROLES = ["textbook", "note", "web"]       # the only roles allowed to support an answer

QWEN_INSTRUCT = "Instruct: Given a student's question, retrieve textbook passages that answer it\nQuery: "
EMBEDDERS = {
    "qwen3-embedding:0.6b": dict(query_prefix=QWEN_INSTRUCT, doc_prefix=""),
    "bge-m3":               dict(query_prefix="", doc_prefix=""),
    "nomic-embed-text":     dict(query_prefix="search_query: ", doc_prefix="search_document: "),
}
EMBED_MODEL = "qwen3-embedding:0.6b"

## 2 · Loading chunks

In [2]:
def load_chunks():
    chunks = [json.loads(l) for l in CHUNKS_FILE.read_text(encoding="utf-8").splitlines()]
    settings = json.loads(CHUNK_SETTINGS_FILE.read_text())
    return chunks, settings


chunks, chunk_settings = load_chunks()
print(len(chunks), "chunks; chunk settings hash", chunk_settings["hash"])
print(Counter(c["role"] for c in chunks))

476 chunks; chunk settings hash eebcd81a8021
Counter({'textbook': 475, 'note': 1})


## 3 · Embedding through Ollama
`/api/embed` takes a batch of texts. The model is unloaded with `keep_alive=0` when we are done, so it doesn't hold VRAM that the LLM will need.

In [3]:
def embed_texts(texts, model, prefix="", batch=EMBED_BATCH, keep_alive="5m"):
    out = []
    for i in range(0, len(texts), batch):
        r = requests.post(f"{OLLAMA_URL}/api/embed", timeout=300,
                          json={"model": model, "input": [prefix + t for t in texts[i:i + batch]], "keep_alive": keep_alive})
        r.raise_for_status()
        out += r.json()["embeddings"]
    v = np.asarray(out, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def embed_query(q, model=EMBED_MODEL):
    return embed_texts([q], model, EMBEDDERS[model]["query_prefix"])[0]


def unload(model):
    requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": [], "keep_alive": 0}, timeout=60)


def model_slug(model):
    return re.sub(r"[^a-z0-9]+", "_", model.lower()).strip("_")

## 4 · Embedding cache
One `.npz` file per source: `<source file hash>_<chunk settings hash>.npz` under the model's folder. Each file also stores a hash of every chunk's text, so a cached vector is only reused for exactly the same text.

In [4]:
def cache_path(model, doc_hash, settings_hash):
    return CACHE_DIR / model_slug(model) / f"{doc_hash[:16]}_{settings_hash}.npz"


def text_hash(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def embed_chunks(chunks, model, settings_hash):
    """Return {chunk_id: vector}, embedding only what the cache doesn't have. Also returns counts."""
    prefix = EMBEDDERS[model]["doc_prefix"]
    by_doc = {}
    for c in chunks:
        by_doc.setdefault(c["doc_hash"], []).append(c)
    vectors, stats = {}, Counter()
    for doc_hash, cs in by_doc.items():
        path = cache_path(model, doc_hash, settings_hash)
        cached = {}
        if path.exists():
            z = np.load(path, allow_pickle=False)
            cached = {(i, h): v for i, h, v in zip(z["ids"], z["text_hashes"], z["vectors"])}
        hashes = [text_hash(c["text"]) for c in cs]
        todo = [(c, h) for c, h in zip(cs, hashes) if (c["chunk_id"], h) not in cached]
        if todo:
            new = embed_texts([c["text"] for c, _ in todo], model, prefix)
            cached.update({(c["chunk_id"], h): v for (c, h), v in zip(todo, new)})
        stats["embedded"] += len(todo)
        stats["from_cache"] += len(cs) - len(todo)
        vecs = np.stack([cached[(c["chunk_id"], h)] for c, h in zip(cs, hashes)])
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(path, ids=np.array([c["chunk_id"] for c in cs]), text_hashes=np.array(hashes), vectors=vecs)
        vectors.update({c["chunk_id"]: v for c, v in zip(cs, vecs)})
    return vectors, stats

## 5 · Chroma collection
One collection per embedding model. The collection records the chunk settings hash; if that changes, the collection is rebuilt from scratch so old chunk IDs can't linger. Metadata holds everything from the chunk except its text, which is stored as the document.

In [5]:
META_KEYS = ["chunk_id", "source_id", "source_name", "role", "page_start", "page_end",
             "title", "has_code", "has_table", "n_tokens", "doc_hash"]


def collection_name(model):
    return f"chunks_{model_slug(model)}"


def build_chroma(chunks, vectors, model, settings_hash):
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    name = collection_name(model)
    existing = {c.name for c in client.list_collections()}
    if name in existing and client.get_collection(name).metadata.get("chunk_settings") != settings_hash:
        client.delete_collection(name)
    col = client.get_or_create_collection(name, metadata={"hnsw:space": "cosine", "embed_model": model,
                                                          "chunk_settings": settings_hash})
    ids = [c["chunk_id"] for c in chunks]
    stale = set(col.get(include=[])["ids"]) - set(ids)
    if stale:
        col.delete(ids=list(stale))
    for i in range(0, len(chunks), 500):
        part = chunks[i:i + 500]
        col.upsert(ids=[c["chunk_id"] for c in part],
                   embeddings=np.stack([vectors[c["chunk_id"]] for c in part]).tolist(),
                   documents=[c["text"] for c in part],
                   metadatas=[{k: c[k] for k in META_KEYS} for c in part])
    return col, len(stale)

## 6 · BM25
Tokens: lowercase words and identifiers (`a-z`, digits, `_`), so code names like `read_csv` or `__init__` stay one token. A short stopword list keeps function words from dominating on these slides.

In [6]:
STOPWORDS = set("""a an the of to in on for and or is are was were be been it its this that these those with as by at from
into than then so if not no do does did can could will would should may might what which who whom how why when where
there their they them we you your our i he she his her""".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9_]+", text.lower()) if t not in STOPWORDS]


def build_bm25(chunks, settings_hash):
    tokens = [tokenize(c["text"]) for c in chunks]
    bm25 = BM25Okapi(tokens)
    with open(BM25_FILE, "wb") as f:
        pickle.dump(dict(bm25=bm25, chunk_ids=[c["chunk_id"] for c in chunks], chunk_settings=settings_hash), f)
    return bm25


def load_bm25():
    with open(BM25_FILE, "rb") as f:
        return pickle.load(f)

## 7 · Build everything

In [7]:
def build_index(model=EMBED_MODEL, unload_after=True):
    chunks, settings = load_chunks()
    t0 = time.time()
    vectors, stats = embed_chunks(chunks, model, settings["hash"])
    t_embed = time.time() - t0
    col, n_stale = build_chroma(chunks, vectors, model, settings["hash"])
    bm25 = build_bm25(chunks, settings["hash"])
    if unload_after:
        unload(model)
    return dict(collection=col, bm25=bm25, stats=stats, seconds=round(t_embed, 1), stale_removed=n_stale)


first = build_index()
print("first build: ", first["stats"], f"{first['seconds']} s", "stale removed", first["stale_removed"])
second = build_index()
print("second build:", second["stats"], f"{second['seconds']} s", "stale removed", second["stale_removed"])
col = second["collection"]

first build:  Counter({'from_cache': 476, 'embedded': 0}) 0.1 s stale removed 0
second build: Counter({'from_cache': 476, 'embedded': 0}) 0.1 s stale removed 0


---
# Checks

## 8 · Counts and shapes

In [8]:
got = col.get(include=["embeddings", "metadatas"])
bm = load_bm25()
dims = {len(e) for e in got["embeddings"]}
print(f"chunks {len(chunks)} | chroma {col.count()} | bm25 docs {len(bm['chunk_ids'])} | vector dims {dims}")
print("ids identical in all three:", set(got["ids"]) == {c["chunk_id"] for c in chunks} == set(bm["chunk_ids"]))
norms = np.linalg.norm(np.asarray(got["embeddings"]), axis=1)
print(f"vector norms: min {norms.min():.4f} max {norms.max():.4f}")
print("collection metadata:", col.metadata)
print("roles in chroma:", Counter(m["role"] for m in got["metadatas"]))

chunks 476 | chroma 476 | bm25 docs 476 | vector dims {1024}
ids identical in all three: True
vector norms: min 1.0000 max 1.0000
collection metadata: {'hnsw:space': 'cosine', 'chunk_settings': 'eebcd81a8021', 'embed_model': 'qwen3-embedding:0.6b'}
roles in chroma: Counter({'textbook': 475, 'note': 1})


## 9 · Cache keys change when settings change
Only the key function is called here; nothing is written. A different chunk settings hash or a different model gives a different file, so stale vectors can't be picked up.

In [9]:
c0 = chunks[0]
print(cache_path(EMBED_MODEL, c0["doc_hash"], chunk_settings["hash"]).relative_to(DATA_DIR))
print(cache_path(EMBED_MODEL, c0["doc_hash"], "otherhash000").relative_to(DATA_DIR))
print(cache_path("bge-m3", c0["doc_hash"], chunk_settings["hash"]).relative_to(DATA_DIR))
print("cache files:", len(list((CACHE_DIR / model_slug(EMBED_MODEL)).glob("*.npz"))))

embed_cache\qwen3_embedding_0_6b\0e0fda908c3108bf_eebcd81a8021.npz
embed_cache\qwen3_embedding_0_6b\0e0fda908c3108bf_otherhash000.npz
embed_cache\bge_m3\0e0fda908c3108bf_eebcd81a8021.npz
cache files: 11


## 10 · Self-retrieval
Each of 60 random chunks is used as its own query, by its stored vector. Top-1 should be the chunk itself. This proves the IDs, vectors and metadata line up.

In [10]:
random.seed(3)
sample = random.sample(range(len(got["ids"])), 60)
res = col.query(query_embeddings=[got["embeddings"][i] for i in sample], n_results=1, include=["distances"])
hits = sum(res["ids"][k][0] == got["ids"][i] for k, i in enumerate(sample))
print(f"self top-1: {hits}/{len(sample)}")

self top-1: 60/60


## 11 · Sample questions: dense vs BM25
Top 3 from each, with the hard role filter (only evidence roles). The real scoring (Recall@5, MRR, nDCG@5) comes in 04 with the eval set.

In [11]:
def dense_search(q, k=5, roles=EVIDENCE_ROLES, sources=None):
    where = {"role": {"$in": roles}}
    if sources:
        where = {"$and": [where, {"source_id": {"$in": sources}}]}
    r = col.query(query_embeddings=[embed_query(q).tolist()], n_results=k, where=where, include=["metadatas", "distances"])
    return [(m, 1 - d) for m, d in zip(r["metadatas"][0], r["distances"][0])]


def bm25_search(q, k=5, roles=EVIDENCE_ROLES):
    scores = bm["bm25"].get_scores(tokenize(q))
    by_id = {c["chunk_id"]: c for c in chunks}
    order = np.argsort(-scores)
    out = []
    for i in order:
        if scores[i] <= 0:                       # no query term in the chunk
            break
        c = by_id[bm["chunk_ids"][i]]
        if c["role"] in roles:
            out.append((c, float(scores[i])))
        if len(out) == k:
            break
    return out


def show(results, label):
    for m, s in results:
        print(f"   {label} {s:6.3f}  {m['source_name'][:32]:32} p{m['page_start']:<4} {m['title'][:50]}")


QUESTIONS = [
    "What is the difference between a list and a tuple?",
    "How do I open a file for writing in Python?",
    "What does the elbow method tell you in k-means?",
    "Explain information gain in decision trees",
    "What is the Widrow-Hoff learning rule?",
    "What is a decorator?",
]
t0 = time.time()
for q in QUESTIONS:
    print(f"Q: {q}")
    show(dense_search(q, 3), "dense")
    show(bm25_search(q, 3), "bm25 ")
print(f"\n{len(QUESTIONS)} dense queries in {time.time() - t0:.1f} s (includes model load)")

Q: What is the difference between a list and a tuple?
   dense  0.724  Ch1 and 2 Introduction to Python p91   Advantages of Tuple over List
   dense  0.697  Ch1 and 2 Introduction to Python p86   Tuples are more efficient
   dense  0.684  Ch1 and 2 Introduction to Python p97   Tuple Assignments
   bm25   9.562  Ch1 and 2 Introduction to Python p91   Advantages of Tuple over List
   bm25   9.447  Ch1 and 2 Introduction to Python p97   Tuple Assignments
   bm25   9.407  Ch1 and 2 Introduction to Python p96   Join tuple elements in a list
Q: How do I open a file for writing in Python?
   dense  0.713  Ch 5 File Handling.pdf           p11   Writing Files
   dense  0.710  Ch 5 File Handling.pdf           p9    Example
   dense  0.695  Ch 5 File Handling.pdf           p14   Writing File using with statement
   bm25  16.466  Ch 5 File Handling.pdf           p14   Writing File using with statement
   bm25  15.232  Ch 5 File Handling.pdf           p11   Writing Files
   bm25  15.170  Ch 5 File 

## 12 · Filters are applied at search time

In [12]:
py_files = [c["source_id"] for c in chunks if c["source_name"].startswith("Ch 5 File")][:1]
r = dense_search("What is a decorator?", 3, sources=py_files)
print("restricted to", py_files, "->", {m["source_id"] for m, _ in r})
r = dense_search("What is a decorator?", 3, roles=["pyq"])
print("roles=['pyq'] ->", len(r), "results (no pyq sources ingested yet)")

restricted to ['458032f3b701'] -> {'458032f3b701'}
roles=['pyq'] -> 0 results (no pyq sources ingested yet)


## 13 · VRAM released

In [13]:
unload(EMBED_MODEL)
print(requests.get(f"{OLLAMA_URL}/api/ps").json()["models"] or "no models loaded")

[{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'size': 323150151, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16'}, 'expires_at': '2026-10-25T13:08:42.7611757+05:30', 'size_vram': 323150151, 'context_length': 2048}, {'name': 'qwen3-vl:4b-instruct-q4_K_M', 'model': 'qwen3-vl:4b-instruct-q4_K_M', 'size': 3527545978, 'digest': 'ee4b975b58c17ce268cd19d40db35d5edc64603035d2ffc1fee1968eb0947f7b', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen3vl', 'families': ['qwen3vl'], 'parameter_size': '4.4B', 'quantization_level': 'Q4_K_M'}, 'expires_at': '2026-10-25T13:05:05.3528805+05:30', 'size_vram': 3527545978, 'context_length': 4096}]
